In [0]:
CREATE OR REPLACE TABLE claims.analytics.weekly_rejection_rate
USING DELTA
AS


SELECT
    pl.plan_code AS plan_code,


    d.week_start AS week_start,


    CAST(
        COUNT_IF(
            f.status IN ('approved', 'rejected', 'settled')
        )
        AS INT
    ) AS decided_count,


    CAST(
        COUNT_IF(f.status = 'rejected')
        AS INT
    ) AS rejected_count,


    ROUND(
        COUNT_IF(f.status = 'rejected') * 1.0
        /
        NULLIF(
            COUNT_IF(
                f.status IN ('approved', 'rejected', 'settled')
            ),
            0
        ),
        4
    ) AS rejection_rate


FROM claims.analytics.fact_claims f
JOIN claims.analytics.dim_plan pl ON f.plan_sk = pl.plan_sk
JOIN claims.analytics.dim_date d ON f.submitted_date_key = d.date_key


WHERE f.status IN ('approved', 'rejected', 'settled')


GROUP BY
    pl.plan_code,
    d.week_start;